In [1]:
%pip install rank-bm25

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import re
import csv
from pathlib import Path

In [3]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [4]:
from scipy.sparse import load_npz
from rank_bm25 import BM25Okapi

In [13]:
DATASET_FOLDER = Path("/Users/megan/Downloads/PMC-Patients-full-v2")

In [14]:
OUTPUT_FOLDER = DATASET_FOLDER / "tfidf_output3"
MATRIX_FILE = OUTPUT_FOLDER / "tfidf_matrix3.npz"
METADATA_FILE = OUTPUT_FOLDER / "document_metadata3.csv"
OUTPUT_FILE = OUTPUT_FOLDER / "task2_similar_patients3.json"

In [15]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [16]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    words = text.split()

    processed_words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(processed_words)

In [17]:
records = []
documents = []

In [18]:
json_files = sorted(DATASET_FOLDER.rglob("*.json"))

In [19]:
for file_path in json_files:

    if OUTPUT_FOLDER in file_path.parents:
        continue

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    items = data if isinstance(data, list) else [data]

    for record in items:
        if not isinstance(record, dict):
            continue

        title = record.get("title", "")
        patient_text = record.get("patient", "")

        if not isinstance(title, str):
            title = ""

        if not isinstance(patient_text, str):
            patient_text = ""

        original_text = f"{title} {patient_text}".strip()

        if not original_text:
            continue

        processed_text = preprocess_text(original_text)

        records.append({
            "patient_uid": str(record.get("patient_uid", "")),
            "patient_id": str(record.get("patient_id", "")),
            "filename": str(
                file_path.relative_to(DATASET_FOLDER)
            )
        })

        documents.append(processed_text)

In [20]:
tfidf_matrix = load_npz(MATRIX_FILE)

In [21]:
with open(METADATA_FILE, "r", encoding="utf-8") as file:
    metadata = list(csv.DictReader(file))

In [22]:
if len(records) != tfidf_matrix.shape[0]:
    raise ValueError(
        "The number of records does not match the TF-IDF "
        "matrix rows. Check the dataset order and Task 1 files."
    )

In [23]:
if len(metadata) != len(records):
    raise ValueError(
        "Metadata row count does not match the records."
    )

In [24]:
for i, record in enumerate(records):
    if record["filename"] != metadata[i]["filename"]:
        raise ValueError(
            f"Record order mismatch at row {i}. "
            "Do not use these results until the order is fixed."
        )

In [25]:
if any(not record["patient_uid"] for record in records):
    raise ValueError("Some records are missing patient_uid.")

In [26]:
patient_uids = [record["patient_uid"] for record in records]

In [27]:
if len(set(patient_uids)) != len(patient_uids):
    raise ValueError(
        "Duplicate patient_uid values found. "
        "The output dictionary requires unique keys."
    )

In [28]:
number_of_patients = len(records)

In [30]:
# only the top five similar patient documents
if number_of_patients < 6:
    raise ValueError("At least six records are needed to retrieve five other patients for every query.")

In [31]:
print("Records:", number_of_patients)
print("TF-IDF matrix:", tfidf_matrix.shape)

Records: 167034
TF-IDF matrix: (167034, 257703)


In [32]:
# TF-IDF retrieval using cosine similarity first
tfidf_results = {}

In [35]:
# don't want to process all at once due to large corpus
batch_size = 500

In [37]:
from scipy.sparse import issparse

for start in range(0, number_of_patients, batch_size):
    end = min(start + batch_size, number_of_patients)

    # Compute similarity for current batch against all patients
    similarity_batch = (tfidf_matrix[start:end] @ tfidf_matrix.T)
    
    if issparse(similarity_batch):
        similarity_batch = similarity_batch.toarray()

    for local_index, scores in enumerate(similarity_batch):
        query_index = start + local_index

        scores[query_index] = float("-inf")

        top_indices = scores.argpartition(-5)[-5:]
        top_indices = top_indices[scores[top_indices].argsort()[::-1]]

        tfidf_results[query_index] = [
            patient_uids[i] for i in top_indices
        ]

    # monitor if it's working
    print(f"TF-IDF processed: {end}/{number_of_patients} ({100*end//number_of_patients}%)")

TF-IDF processed: 500/167034 (0%)
TF-IDF processed: 1000/167034 (0%)
TF-IDF processed: 1500/167034 (0%)
TF-IDF processed: 2000/167034 (1%)
TF-IDF processed: 2500/167034 (1%)
TF-IDF processed: 3000/167034 (1%)
TF-IDF processed: 3500/167034 (2%)
TF-IDF processed: 4000/167034 (2%)
TF-IDF processed: 4500/167034 (2%)
TF-IDF processed: 5000/167034 (2%)
TF-IDF processed: 5500/167034 (3%)
TF-IDF processed: 6000/167034 (3%)
TF-IDF processed: 6500/167034 (3%)
TF-IDF processed: 7000/167034 (4%)
TF-IDF processed: 7500/167034 (4%)
TF-IDF processed: 8000/167034 (4%)
TF-IDF processed: 8500/167034 (5%)
TF-IDF processed: 9000/167034 (5%)
TF-IDF processed: 9500/167034 (5%)
TF-IDF processed: 10000/167034 (5%)
TF-IDF processed: 10500/167034 (6%)
TF-IDF processed: 11000/167034 (6%)
TF-IDF processed: 11500/167034 (6%)
TF-IDF processed: 12000/167034 (7%)
TF-IDF processed: 12500/167034 (7%)
TF-IDF processed: 13000/167034 (7%)
TF-IDF processed: 13500/167034 (8%)
TF-IDF processed: 14000/167034 (8%)
TF-IDF proce

In [42]:
# I picked BM25 for the next method
tokenized_documents = [document.split() for document in documents]

In [43]:
bm25 = BM25Okapi(tokenized_documents)

In [40]:
bm25_results = {}

In [47]:
#BM25 was taking too long, so tried a subset
subset_size = 100 
patient_indices_to_process = range(subset_size)
patient_indices_to_process = range(subset_size)

batch_size = 20

print(f"Starting BM25 processing for {subset_size} patients...")

for query_index in patient_indices_to_process:
    # Print at the start of each iteration to see progress immediately
    if query_index % 10 == 0:  # Print every 10 iterations
        print(f"Processing patient {query_index}...")
    
    query_tokens = tokenized_documents[query_index]
    
    scores = bm25.get_scores(query_tokens)

    # Exclude the query patient.
    scores[query_index] = float("-inf")

    top_indices = scores.argsort()[-5:][::-1]

    bm25_results[query_index] = [patient_uids[i] for i in top_indices]

    # Progress updates
    if (query_index + 1) % batch_size == 0:
        print(
            f"BM25 processed: "
            f"{query_index + 1}/{len(patient_indices_to_process)}")

print(f"Processing complete! Processed {len(patient_indices_to_process)} patients.")

Starting BM25 processing for 100 patients...
Processing patient 0...
Processing patient 10...
BM25 processed: 20/100
Processing patient 20...
Processing patient 30...
BM25 processed: 40/100
Processing patient 40...
Processing patient 50...
BM25 processed: 60/100
Processing patient 60...
Processing patient 70...
BM25 processed: 80/100
Processing patient 80...
Processing patient 90...
BM25 processed: 100/100
Processing complete! Processed 100 patients.


In [48]:
results = {}

In [54]:
# Need to convert to list and get first 100 items, since the BM25 dataset only used 100 patients
if hasattr(records, 'iloc'):
    records = records.iloc[:100]
elif isinstance(records, dict):
    records = list(records.items())[:100] if len(records) > 100 else list(records.items())
else:
    records = list(records)[:100]

if hasattr(tfidf_results, 'iloc'):
    tfidf_results = tfidf_results.iloc[:100]
elif isinstance(tfidf_results, dict):
    tfidf_results = list(tfidf_results.values())[:100] if len(tfidf_results) > 100 else list(tfidf_results.values())
else:
    tfidf_results = list(tfidf_results)[:100]

if hasattr(bm25_results, 'iloc'):
    bm25_results = bm25_results.iloc[:100]
elif isinstance(bm25_results, dict):
    bm25_results = list(bm25_results.values())[:100] if len(bm25_results) > 100 else list(bm25_results.values())
else:
    bm25_results = list(bm25_results)[:100]

results = {}  # Initialize results dictionary if not already defined
for i, record in enumerate(records):
    # Handle if records was converted from dict items (tuples)
    if isinstance(record, tuple):
        record = record[1]  # Get the value from (key, value) tuple
    
    results[record["patient_uid"]] = {
        "patient_id": record["patient_id"],
        "tfidf": tfidf_results[i],
        "bm25": bm25_results[i]
    }

In [55]:
with open(OUTPUT_FILE, "w", encoding="utf-8") as file:
    json.dump(results, file, indent=2, ensure_ascii=False)

In [56]:
print("\nTask 2 completed!")
print("Patients processed:", len(results))
print("Output saved to:", OUTPUT_FILE)


Task 2 completed!
Patients processed: 100
Output saved to: /Users/megan/Downloads/PMC-Patients-full-v2/tfidf_output3/task2_similar_patients3.json


In [57]:
#I wanted to show one example to make sure it looks right 

first_uid = patient_uids[0]
print("\nExample result:")
print(json.dumps(
    {first_uid: results[first_uid]},
    indent=2
))


Example result:
{
  "1065068-1": {
    "patient_id": "88390",
    "tfidf": [
      "8107005-1",
      "5930534-1",
      "8107005-2",
      "5813425-1",
      "2529306-1"
    ],
    "bm25": [
      "6128265-1",
      "5094039-1",
      "4979841-1",
      "5813425-1",
      "8448063-1"
    ]
  }
}
